# pdesolver — benchmark reproduzível

Este notebook reproduz as afirmações de **acurácia**, **capacidades** e **GPU** do artigo.

## O que rodar aqui e o que não rodar

| Medida | Colab é adequado? | Por quê |
|:---|:---|:---|
| RMSE / acurácia | **sim** | determinístico; independe de hardware |
| Matriz de capacidades | **sim** | determinístico |
| CPU vs GPU | **sim** | T4 gratuita; é o único jeito de um revisor sem GPU verificar |
| Escalabilidade (razões) | sim, com ressalva | as razões se mantêm; os absolutos não |
| **Tempo absoluto de CPU** | **não** | VM compartilhada, 2 vCPUs, preempção — variância maior que bare metal ocioso |

Para tempos de CPU publicáveis, rode `benchmarks/benchmark_completo.py` numa máquina **ociosa** e reporte junto o número de calibração que o script imprime.

> **Ative a GPU antes de rodar:** `Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU`

## 1. Instalação

In [ ]:
REPO = 'https://github.com/maiocacedo/PDESsolver'
BRANCH = 'main'

!pip install -q "numpy>=2.0" "scipy>=1.13" "sympy>=1.13" matplotlib
!pip install -q fipy==4.0.2 py-pde==0.56.0
!git clone -q --branch $BRANCH $REPO pdesolver_repo || echo 'clone falhou — veja a celula abaixo'
%cd pdesolver_repo
!pip install -q --no-deps -e .
print('instalado')

Se o repositório ainda não tiver as mudanças desta versão publicadas, suba o código manualmente:
`Arquivos → Upload` e depois `%cd` para a pasta. O notebook precisa de `pdesolver >= 0.2.0` (backends, regiões e análise).

## 2. Ambiente

In [ ]:
import subprocess, sys
sys.path.insert(0, '.')

from benchmarks.benchmark_completo import ambiente, imprime_ambiente
env = ambiente()
imprime_ambiente(env)

print()
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                          '--format=csv'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('sem GPU — ative T4 em Ambiente de execucao')

## 3. Matriz de capacidades

O que cada biblioteca consegue **expressar na própria notação**. Onde duas conseguem, o teste verifica que **concordam** — não elege vencedor.

In [ ]:
!python benchmarks/benchmark_completo.py --parte capacidades

## 4. Acurácia nos problemas de referência

O RMSE é determinístico e deve reproduzir os valores do artigo exatamente.

**Atenção ao esquema do FiPy.** `TransientTerm() == DiffusionTerm(...)` resolvido com `eq.solve()` é **Euler implícito** (1ª ordem no tempo), não Crank–Nicolson. Comparar um método de 2ª ordem contra ele exagera a diferença de acurácia. A suíte roda os dois e rotula corretamente; a comparação justa é `FiPy/CN`.

Ignore as colunas de tempo desta célula — veja o aviso no topo.

In [ ]:
!python benchmarks/benchmark_completo.py --parte casos --runs 10

## 5. CPU vs GPU

Esta é a seção que o artigo não tem. O operador vetorizado é neutro quanto ao *array module*: o mesmo código roda em NumPy ou CuPy, e os resultados devem ser **idênticos bit a bit**.

In [ ]:
import time
import numpy as np
import matplotlib; matplotlib.use('Agg')
from pdesolver import PDE, PDES

try:
    import cupy as cp
    TEM_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    TEM_GPU = False

def operador(n):
    p = PDE('dF/dt = 0.1*d2F/dx2 + 0.2*d2F/dy2', 'F', ['x', 'y'], ['t'],
            ivar_boundary=[(0, 1), (0, 1)], expr_ic='sin(pi*x)*sin(pi*y)',
            west_bd='Dirichlet', west_func_bd='0', east_bd='Dirichlet', east_func_bd='0',
            north_bd='Dirichlet', north_func_bd='0', south_bd='Dirichlet', south_func_bd='0')
    s = PDES([p], [n, n], backend='stencil')
    s.discretize('central')
    return s.operator, np.array(s.ic)

def cronometra(fn, reps, sync=None):
    for _ in range(5):
        fn()
    if sync: sync()
    melhor = None
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(reps): fn()
        if sync: sync()
        dt = (time.perf_counter() - t0) / reps
        melhor = dt if melhor is None else min(melhor, dt)
    return melhor

print(f"{'N':>6}{'DOF':>10}{'CPU (us)':>12}{'GPU (us)':>12}{'ganho':>9}   max|CPU-GPU|")
for n in (128, 256, 512, 1024):
    op, u = operador(n)
    reps = max(10, int(3e7 // (n * n)))
    t_cpu = cronometra(lambda: op(0.0, u), reps)
    if TEM_GPU:
        gop = op.to_device(cp); gu = cp.asarray(u)
        dif = np.max(np.abs(op(0.0, u) - cp.asnumpy(gop(0.0, gu))))
        t_gpu = cronometra(lambda: gop(0.0, gu), reps,
                           sync=cp.cuda.Stream.null.synchronize)
        print(f'{n:6d}{n*n:10d}{t_cpu*1e6:12.1f}{t_gpu*1e6:12.1f}{t_cpu/t_gpu:8.1f}x   {dif:.2e}')
    else:
        print(f'{n:6d}{n*n:10d}{t_cpu*1e6:12.1f}{"-":>12}{"-":>9}')

## 6. Escalabilidade da montagem

O caminho por nó gera uma expressão simbólica **por ponto de malha**, então seu custo cresce com a malha embora a EDP não mude. O vetorizado faz o trabalho simbólico uma vez.

As **razões** entre as duas colunas transferem entre máquinas; os valores absolutos não.

In [ ]:
!python benchmarks/benchmark_completo.py --parte escala

## 7. Exportar

Gera o JSON com tudo, para anexar ao artigo como material suplementar.

In [ ]:
!python benchmarks/benchmark_completo.py --runs 10 --json resultados_colab.json
from google.colab import files
files.download('resultados_colab.json')